## Business Financial Records Dataset Cleaning

**Dataset:** Buisness Transaction Dataset (100,000 initial records)

---

## Executive Summary

This notebook documents the data cleaning process for a Buisness Transacton dataset. The dataset contains information about 100,000 transactions.

**Key Cleaning Operations:**
- Removed 994 duplicate customer records
- Handled Date column 
- Handled missing values by using statistical imputation
- Handled missing values by dropping column with too many missing values
- Handled some columns containing distorted values,abbrevated and incorrectly spelt words 
- Handled negative numerical values by grouping according to transction_status 

- Preserved data integrity while maintaining business logic

**Final Output:** A clean dataset of 28,399 unique transactions ready for exploratory data analysis

---
## 1. Environment Setup

We begin by importing the necessary Python libraries for data manipulation and analysis. Warnings are suppressed to maintain clean output throughout the notebook.

In [1]:
import numpy as np
import pandas as pd
import warnings
import re
warnings.filterwarnings('ignore')

---
## 2. Data Loading & Initial Inspection

The raw dataset is loaded from CSV format. This initial inspection helps us understand the data structure, identify potential quality issues, and plan our cleaning strategy.


In [2]:
df =pd.read_csv(r"c:\Users\HP USER\Downloads\dirty_financial_transactions.csv")

Inspecting the dataset

In [3]:
df.shape

(100000, 8)

The dataset contains **100,000 rows** (customer records) and **8 columns** (features). This is our starting point before any cleaning operations.

In [4]:
df.head(20)

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,2025-02-30,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,2025-02-30,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed
5,T0006,2021-10-26,C4241,Smartphone,598.0,NaN,PayPal,Completed
6,NaN,2025-02-30,C1313,Laptop,10.0,NaN,credit card,Completed
7,T0008,2023-13-01,C4736,Headphones,669.0,-86.92126929493884,Cash,NaN
8,T0009,NaN,C3387,Tablet,10.0,461.70198437439694,PayPal,NaN
9,T0010,2025-02-30,C2846,Laptop,-1.0,404.8907066405689,creditcard,Pending


**Initial Observations:**
- Missing values are visible in `Transaction ID` 
- `Transaction Date` column are not in properly aligned format and they also contain some missing values
- `Product_Name` contains alot of words spelt incorrectly and alot of inconsistent casing
- `Quantity` column contains visible missing values and quite a number of negative values
- `Price` column contains quite a number of visible missing values and negative values
- Inconsistent casing are visible across the `Payment_Method` column
- Missing values and inconsistent casing are visible across the `Transaction_Status` column

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Transaction_ID      94982 non-null   object 
 1   Transaction_Date    95120 non-null   object 
 2   Customer_ID         95122 non-null   object 
 3   Product_Name        100000 non-null  object 
 4   Quantity            94981 non-null   float64
 5   Price               66503 non-null   object 
 6   Payment_Method      100000 non-null  object 
 7   Transaction_Status  83321 non-null   object 
dtypes: float64(1), object(7)
memory usage: 6.1+ MB


---

## 3. Handling Duplicate Records

Duplicate Transaction records can skew our analysis . We identify and remove exact duplicates across all columns to ensure each one appears only once in our dataset.

In [6]:
df.duplicated().sum()

np.int64(994)

### Lets take a look at the duplicates records in our dataset

In [7]:
df[df.duplicated()]

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
3544,T1579,NaN,C2854,Headphones,4.0,978.806380519704,credit card,completed
4816,T4817,2023-13-01,C756,Laptop,170.0,-461.44380335792755,Cash,NaN
5455,T5456,2022-11-01,C1548,Tablet,530.0,-706.4750815495283,PayPal,Pending
5938,T5939,2023-04-14,C2656,Laptop,6.0,NaN,Cash,complete
8125,T8126,2021-05-25,C2396,Smartphone,-4.0,335.9920520635594,PayPal,completed
...,...,...,...,...,...,...,...,...
99674,T47093,2023-13-01,C4823,Tablet,-6.0,-522.4003199263182,creditcard,Completed
99786,T15545,2023-13-01,C2823,Laptop,3.0,NaN,pay pal,completed
99866,T6571,2023-13-01,C4483,Smartphone,2.0,421.99494928905284,PayPal,completed
99984,NaN,2022-01-26,C4753,Laptop,480.0,-558.7481358840021,Cash,completed


In [8]:
df.drop_duplicates(inplace=True)

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.shape

(99006, 8)

**Result:** The dataset now contains **99,006 unique records** (reduced from 100,000), indicating that **994 duplicate rows** were successfully removed.

### Fixing the inconsistencies in the date column

In [11]:
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

In [12]:
df['Transaction_Date']= pd.to_datetime(df['Transaction_Date'], infer_datetime_format=True, errors='coerce')

In [13]:
df['Transaction_Date']= pd.to_datetime(df['Transaction_Date'], dayfirst=True, errors='coerce')

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99006 entries, 0 to 99998
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      94040 non-null  object        
 1   Transaction_Date    31440 non-null  datetime64[ns]
 2   Customer_ID         94176 non-null  object        
 3   Product_Name        99006 non-null  object        
 4   Quantity            94034 non-null  float64       
 5   Price               65875 non-null  object        
 6   Payment_Method      99006 non-null  object        
 7   Transaction_Status  82485 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 6.8+ MB


In [15]:
df.isnull().sum()/len(df)*100

Transaction_ID         5.015858
Transaction_Date      68.244349
Customer_ID            4.878492
Product_Name           0.000000
Quantity               5.021918
Price                 33.463628
Payment_Method         0.000000
Transaction_Status    16.686867
dtype: float64

The `Transaction_Date` column contained over 60% missing values after conversion to datetime format.

Dropping rows would result in significant data loss, while imputing dates could introduce inaccurate temporal patterns.

Therefore, the column was removed to maintain data quality and avoid misleading time-based analysis.

Dropping the date column

In [16]:
df = df.drop(columns=['Transaction_Date'])

In [17]:
df.head()

,Transaction_ID,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed


The date column has been sucessfully dropped.

In [18]:
df.isna().sum()

Transaction_ID         4966
Customer_ID            4830
Product_Name              0
Quantity               4972
Price                 33131
Payment_Method            0
Transaction_Status    16521
dtype: int64

In [19]:
df.isnull().sum()/len(df)*100

Transaction_ID         5.015858
Customer_ID            4.878492
Product_Name           0.000000
Quantity               5.021918
Price                 33.463628
Payment_Method         0.000000
Transaction_Status    16.686867
dtype: float64

---
## 4. Missing Value Analysis

Let's examine the state of the dataset more closely. Understanding the distribution and characteristics of our features helps inform the best approach for handling missing values.

### Handling Missing values in `Customer_ID` Column

In [20]:
df['Customer_ID'] = df['Customer_ID'].fillna('Unknown')

The `customer_id` column contained missing (null) values, which could affect data consistency during analysis.

Instead of dropping these rows or the entire column, the missing values were replaced with `"Unknown"`

This approach was chosen because:
- Dropping rows would lead to unnecessary data loss.
- Filling with `"Unknown"` preserves all records while clearly indicating unidentified customers.

This ensures the dataset remains complete and usable without introducing misleading or fabricated customer information.

### Handling Missing values in `Transaction_ID` Column

In [21]:
df = df.dropna(subset=['Transaction_ID'])

The `transaction_id` column contained missing (null) values. Since this column serves as a unique identifier for each transaction, it is essential for maintaining data integrity.

Rows with missing `transaction_id` values were removed.

This approach was chosen because:
- Transaction IDs cannot be reliably inferred or filled without introducing errors.
- Keeping null values could lead to duplicate or untraceable records.
- Removing only the affected rows preserves the overall structure and reliability of the dataset.

This ensures that each remaining record in the dataset is uniquely identifiable and suitable for accurate analysis.

---
## 5.Handling Inconsistencies

#### Handling Inconsistent Words and  Casing in `Product_Name` Column

In [22]:
df['Product_Name'].unique()

array(['Headphones', 'Coffee ', 'Tablet', 'Tab', 'Coffee Machine',
       'Smartphone', 'Laptop', 'Coffee Ma', 'Cof', 'Smar', 'Coffee M',
       'T', 'Smartp', 'Headp', 'Smart', 'La', 'Lapt', 'Tabl', 'L', 'C',
       'Smartph', 'Hea', 'Head', 'Lapto', 'Headphon', 'Table', 'Co',
       'Headphone', 'Coffee Mac', 'Sm', 'Coffee', 'Headph', 'S',
       'Coffee Mach', 'Smartphon', 'Headpho', 'Coffee Machin', 'Coff',
       'Smartpho', 'Lap', 'H', 'Ta', 'Coffee Machi', 'He', 'Coffe', 'Sma'],
      dtype=object)

In [23]:
df['Product_Name'] = df['Product_Name'].replace({
    'C': 'Coffee',
    'Co': 'Coffee',
    'Cof': 'Coffee',
    'Coff': 'Coffee',
    'Coffe': 'Coffee',
    'Coffee M': 'Coffee Machine',
    'Coffee Ma' :'Coffee Machine',
    'Coffee Mac' :'Coffee Machine',
    'Coffee Mach' :'Coffee Machine',
    'Coffee Machi' :'Coffee Machine',
    'Coffee Machin' :'Coffee Machine',
    'S' : 'Smartphone',
    'Sm' :'Smartphone',
    'Sma':'Smartphone',
    'Smar': 'Smartphone',
    'Smart': 'Smartphone',
    'Smartp':'Smartphone',
    'Smartph':'Smartphone',
    'Smartpho':'Smartphone',
    'Smartphon':'Smartphone',
    'T': 'Tablet',
    'Ta': 'Tablet',
    'Tab': 'Tablet',
    'Tabl': 'Tablet',
    'Table': 'Tablet',
    'H': 'Headphones',
    'He': 'Headphones',
    'Hea': 'Headphones',
    'Head': 'Headphones',
    'Headp': 'Headphones',
    'Headph': 'Headphones',
    'Headpho': 'Headphones',
    'Headphon': 'Headphones',
    'Headphone': 'Headphones',
    'L': 'Laptop',
    'La': 'Laptop',
    'Lap': 'Laptop',
    'Lapt': 'Laptop',
    'Lapto': 'Laptop',
})

In [24]:
df['Product_Name'] = df['Product_Name'].str.strip().str.title()

#### Handling Inconsistent Words and  Casing in `Payment_Method` Column

In [25]:
df['Payment_Method'].unique()

array(['pay pal', 'creditcard', 'credit card', 'PayPal', 'Cash',
       'PayPal ', 'Credit Card'], dtype=object)

In [26]:
df['Payment_Method'].unique()

array(['pay pal', 'creditcard', 'credit card', 'PayPal', 'Cash',
       'PayPal ', 'Credit Card'], dtype=object)

In [27]:
df['Payment_Method'] = df['Payment_Method'].replace({
  'Pay Pal':'Paypal',
  'Creditcard':'Credit Card', 
  'creditcard' :'Credit Card',
  'paypal':'Paypal',
})

In [28]:
df['Payment_Method'] = df['Payment_Method'].str.strip().str.title()

In [29]:
df['Transaction_Status'].unique()

array([nan, 'Pending', 'completed', 'Completed', 'complete', 'Failed'],
      dtype=object)

In [30]:
df['Transaction_Status'] = df['Transaction_Status'].replace({
   'Complete':'Completed', 
   'complete': 'Completed',
   })

In [31]:
df['Transaction_Status'] = df['Transaction_Status'].str.strip().str.title()

In [32]:
df['Transaction_Status'] = df['Transaction_Status'].fillna(df['Transaction_Status'].mode()[0])

In [33]:
df.head()

,Transaction_ID,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,C2205,Headphones,-5.0,$420.21,Pay Pal,Completed
1,T0002,C3156,Coffee,469.0,-445.34202525395585,Credit Card,Pending
2,T0003,C2919,Tablet,-4.0,810.9930123946459,Credit Card,Completed
3,T0004,C3009,Tablet,-7.0,868.6083413217348,Paypal,Pending
4,T0005,C3488,Coffee Machine,-10.0,-763.1224490039416,Paypal,Completed


### Handling the missing values and negative values in the two major numerical colomn
- `Price`
- `Quantity` 

In [34]:
df.isna().sum()

Transaction_ID            0
Customer_ID               0
Product_Name              0
Quantity               4676
Price                 31429
Payment_Method            0
Transaction_Status        0
dtype: int64

In [35]:
df.isnull().sum()/len(df)*100

Transaction_ID         0.000000
Customer_ID            0.000000
Product_Name           0.000000
Quantity               4.972352
Price                 33.420885
Payment_Method         0.000000
Transaction_Status     0.000000
dtype: float64

The `Price` column contains some values that has the dollar sign attached to it,

We'll fix that by removing the .
This approach maintains data integrity .

In [36]:
df['Price'] = (
    df['Price']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
)

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94040 entries, 0 to 99998
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Transaction_ID      94040 non-null  object 
 1   Customer_ID         94040 non-null  object 
 2   Product_Name        94040 non-null  object 
 3   Quantity            89364 non-null  float64
 4   Price               94040 non-null  object 
 5   Payment_Method      94040 non-null  object 
 6   Transaction_Status  94040 non-null  object 
dtypes: float64(1), object(6)
memory usage: 5.7+ MB


It is observed that the data type of `Price` and `Quantity` is incorrect

So we change them to the appropriate data type

In [38]:
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94040 entries, 0 to 99998
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Transaction_ID      94040 non-null  object 
 1   Customer_ID         94040 non-null  object 
 2   Product_Name        94040 non-null  object 
 3   Quantity            89364 non-null  float64
 4   Price               62611 non-null  float64
 5   Payment_Method      94040 non-null  object 
 6   Transaction_Status  94040 non-null  object 
dtypes: float64(2), object(5)
memory usage: 5.7+ MB


The data type has been adjusted to the correct one

## 6.Imputation Strategy:

Given the minimal number of missing values in both rows, we'll use statistical imputation methods that preserve the distribution characteristics of each feature:

**Numeric Features:** Use **median imputation** rather than mean to reduce the impact of potential outliers.


This approach maintains data integrity while avoiding information loss from row deletion.

In [40]:
for col in df.select_dtypes(include='number'):
    df[col].fillna(df[col].median(), inplace=True)


In [41]:
df['Quantity'] = df['Quantity'].astype('int')

In [42]:
df.head(10)

,Transaction_ID,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,C2205,Headphones,-5,420.210000,Pay Pal,Completed
1,T0002,C3156,Coffee,469,-445.342025,Credit Card,Pending
2,T0003,C2919,Tablet,-4,810.993012,Credit Card,Completed
3,T0004,C3009,Tablet,-7,868.608341,Paypal,Pending
4,T0005,C3488,Coffee Machine,-10,-763.122449,Paypal,Completed
5,T0006,C4241,Smartphone,598,50.770969,Paypal,Completed
7,T0008,C4736,Headphones,669,-86.921269,Cash,Completed
8,T0009,C3387,Tablet,10,461.701984,Paypal,Completed
9,T0010,C2846,Laptop,-1,404.890707,Credit Card,Pending
10,T0011,C1004,Laptop,10,-600.839309,Paypal,Completed


In [43]:
df_clean = df[
    (df['Quantity'] > 0) & 
    (df['Price'] > 0) & 
    (df['Transaction_Status'] == 'Completed')
]

### Handling Negative Values in Price and Quantity

Negative values were identified in the `Price` and `Quantity` columns. 
Upon inspection, these records were associated with pending or incomplete transactions, which likely represent cancellations or refunds.

For the purpose of accurate sales analysis, these records were excluded to ensure that only completed transactions with valid positive values were analyzed.

This approach ensures that revenue calculations and business insights reflect actual completed sales.

In [44]:
df.shape

(94040, 7)

## 7. Final Data Quality Check

After completing all cleaning steps, we perform a final validation to ensure:

- No missing values remain
- Data types are correct
- Only valid transactions are included

In [45]:
df_clean.shape

(28399, 7)

In [46]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28399 entries, 5 to 99998
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Transaction_ID      28399 non-null  object 
 1   Customer_ID         28399 non-null  object 
 2   Product_Name        28399 non-null  object 
 3   Quantity            28399 non-null  int64  
 4   Price               28399 non-null  float64
 5   Payment_Method      28399 non-null  object 
 6   Transaction_Status  28399 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 1.7+ MB


In [47]:
df_clean.isnull().sum()

Transaction_ID        0
Customer_ID           0
Product_Name          0
Quantity              0
Price                 0
Payment_Method        0
Transaction_Status    0
dtype: int64

In [48]:
df_clean.describe()

,Quantity,Price
count,28399.000000,28399.000000
mean,259.479348,284.012097
std,323.505122,303.751467
min,1.000000,50.028806
25%,6.000000,50.770969
50%,10.000000,50.770969
75%,518.000000,515.800926
max,1000.000000,999.961291


In [49]:
df_clean.to_csv("cleaned_business_financial_records.csv", index=False)

## 8. Conclusion

In this project, we successfully cleaned and prepared a business transaction dataset by:

- Removing duplicates
- Handling missing values using appropriate strategies
- Standardizing categorical variables
- Fixing inconsistent formats
- Filtering out invalid transactions (negative values and incomplete transactions)

A cleaned dataset (`df_clean`) was created to support accurate and reliable analysis of completed business transactions.

This dataset is now ready for exploratory data analysis (EDA) and machine learning applications.